# Notebook 3: Molecular Dynamics Simulation
**ProteinIQ Project**

## Background
Molecular Dynamics (MD) simulates the physical movement of atoms by numerically integrating Newton's equations of motion:

$$\mathbf{F}_i = m_i \ddot{\mathbf{r}}_i = -\nabla_{\mathbf{r}_i} U(\mathbf{r})$$

We implement:
- **Velocity Verlet integrator** — time-reversible, energy-conserving
- **Lennard-Jones potential** — non-bonded interactions
- **Harmonic bond/angle potentials** — covalent geometry
- **Berendsen NVT thermostat** — velocity rescaling to maintain temperature
- **RMSD tracking** — structural deviation from starting conformation

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname('__file__'), '..', 'backend'))
from models.md_simulation import (run_md, trajectory_to_dict,
                                   maxwell_boltzmann_speed)
from models.energy_minimizer import random_coil_coords, residue_charges
np.random.seed(42)

## 1. Maxwell-Boltzmann Initial Velocity Distribution
Before running MD, atoms are assigned velocities sampled from the Maxwell-Boltzmann distribution at the target temperature.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for T, color in [(100,'#378ADD'), (300,'#1D9E75'), (500,'#D85A30')]:
    speeds = maxwell_boltzmann_speed(T)
    ax.hist(speeds, bins=30, alpha=0.6, color=color, label=f'T={T}K',
            density=True, edgecolor='none')
ax.set_xlabel('Speed (Å/ps)', fontsize=11)
ax.set_ylabel('Probability density', fontsize=11)
ax.set_title('Maxwell-Boltzmann speed distributions at different temperatures')
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 2. Run MD Simulation — NVT Ensemble at 300K

In [ ]:
seq    = 'ACDEFGHIKLMN'   # 12-residue peptide
coords = random_coil_coords(len(seq), seed=1)

print(f'Simulating {len(seq)}-residue chain...')
print(f'Timestep : {0.002} ps (2 fs)')
print(f'Steps    : 1000  →  total time = {1000*0.002:.2f} ps')
print(f'Ensemble : NVT (Berendsen thermostat, τ=0.1 ps, T=300K)\n')

traj, final_state = run_md(
    coords,
    T_target   = 300.0,
    n_steps    = 1000,
    save_every = 10,
    seed       = 42,
)

print(f'Recorded {len(traj.steps)} frames')
print(f'Mean temperature : {np.mean(traj.temperatures):.1f} K')
print(f'Final RMSD       : {traj.rmsds[-1]:.2f} Å')

## 3. Trajectory Analysis — Temperature, Energy, RMSD

In [ ]:
fig = plt.figure(figsize=(14, 9))
gs  = GridSpec(3, 2, figure=fig, hspace=0.4, wspace=0.35)

# Temperature
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(traj.times, traj.temperatures, color='#D85A30', lw=1.5, label='Instantaneous T')
ax1.axhline(300, color='gray', ls='--', lw=1.2, label='Target 300K')
ax1.fill_between(traj.times, traj.temperatures, 300, alpha=0.1, color='#D85A30')
ax1.set_ylabel('Temperature (K)')
ax1.set_xlabel('Time (ps)')
ax1.set_title('Temperature fluctuations — Berendsen NVT thermostat')
ax1.legend()
ax1.grid(alpha=0.2)

# Potential & Kinetic energy
ax2 = fig.add_subplot(gs[1, 0])
ax2.plot(traj.times, traj.energies_pe, color='#378ADD', lw=1.5, label='Potential')
ax2.plot(traj.times, traj.energies_ke, color='#1D9E75', lw=1.5, label='Kinetic')
ax2.set_ylabel('Energy (kcal/mol)')
ax2.set_xlabel('Time (ps)')
ax2.set_title('Energy components over simulation')
ax2.legend()
ax2.grid(alpha=0.2)

# Total energy
ax3 = fig.add_subplot(gs[1, 1])
ax3.plot(traj.times, traj.energies_total, color='#D85A30', lw=1.5)
ax3.fill_between(traj.times, traj.energies_total, alpha=0.1, color='#D85A30')
ax3.set_ylabel('Total energy (kcal/mol)')
ax3.set_xlabel('Time (ps)')
ax3.set_title('Total energy (E_kin + E_pot)')
ax3.grid(alpha=0.2)

# RMSD
ax4 = fig.add_subplot(gs[2, :])
ax4.plot(traj.times, traj.rmsds, color='#1D9E75', lw=2)
ax4.fill_between(traj.times, traj.rmsds, alpha=0.12, color='#1D9E75')
ax4.set_ylabel('RMSD (Å)')
ax4.set_xlabel('Time (ps)')
ax4.set_title('RMSD from initial structure — structural flexibility over time')
ax4.grid(alpha=0.2)

plt.suptitle(f'MD Trajectory Analysis — {seq} at 300K', fontsize=13, fontweight='bold', y=1.01)
plt.savefig('../data/figures/md_trajectory.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Temperature Equilibration — Comparing 3 Targets

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, T_target, color in zip(axes, [200,300,500], ['#378ADD','#1D9E75','#D85A30']):
    traj_t, _ = run_md(coords, T_target=T_target, n_steps=800,
                        save_every=8, seed=42)
    ax.plot(traj_t.times, traj_t.temperatures, color=color, lw=1.5)
    ax.axhline(T_target, color='gray', ls='--', lw=1.2)
    ax.fill_between(traj_t.times, traj_t.temperatures, T_target,
                    alpha=0.1, color=color)
    ax.set_title(f'T_target = {T_target}K\nMean = {np.mean(traj_t.temperatures):.0f}K')
    ax.set_xlabel('Time (ps)')
    ax.set_ylabel('Temperature (K)')
    ax.grid(alpha=0.2)
plt.suptitle('Berendsen thermostat equilibration', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Velocity Verlet Algorithm — Illustrated

In [ ]:
# Demonstrate Verlet integration on a simple 1D harmonic oscillator
dt, k, m = 0.01, 1.0, 1.0
x, v = 1.0, 0.0   # start at max displacement
times, xs, vs, Es = [], [], [], []
for i in range(500):
    F       = -k * x
    a       = F / m
    x_new   = x + v*dt + 0.5*a*dt**2
    F_new   = -k * x_new
    a_new   = F_new / m
    v_new   = v + 0.5*(a+a_new)*dt
    x, v    = x_new, v_new
    KE = 0.5*m*v**2
    PE = 0.5*k*x**2
    times.append(i*dt); xs.append(x); vs.append(v); Es.append(KE+PE)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].plot(times, xs, color='#378ADD', lw=1.5)
axes[0].set_title('Position vs time\n(harmonic oscillator)'); axes[0].set_xlabel('t'); axes[0].set_ylabel('x')
axes[0].grid(alpha=0.2)
axes[1].plot(times, vs, color='#D85A30', lw=1.5)
axes[1].set_title('Velocity vs time'); axes[1].set_xlabel('t'); axes[1].set_ylabel('v')
axes[1].grid(alpha=0.2)
axes[2].plot(times, Es, color='#1D9E75', lw=1.5)
axes[2].axhline(np.mean(Es), color='gray', ls='--', lw=1)
axes[2].set_title(f'Total energy conservation\n(drift < {(max(Es)-min(Es)):.4f})')
axes[2].set_xlabel('t'); axes[2].set_ylabel('E')
axes[2].grid(alpha=0.2)
plt.suptitle('Velocity Verlet integrator — energy conservation check', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Max energy drift: {max(Es)-min(Es):.6f} (excellent conservation)')